# Explore outputs from Exercises 1–6

This notebook validates and visualizes your output without revealing hidden labels or private leaderboard scores. Select an exercise, run all cells, and upload your CSV when prompted.

In [ ]:
EXERCISE = "01 — HomeValue" # @param ["01 — HomeValue", "02 — Retention", "03 — BookingGuard", "04 — Segment Studio", "05 — VisualSort", "06 — Causal Campaign"]
USE_PUBLIC_SAMPLE = False # @param {type:"boolean"}
print(EXERCISE, '| use sample:', USE_PUBLIC_SAMPLE)

In [ ]:
from pathlib import Path
import subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

REPO=Path('/content/modern-ai-ml-exercises')
if not REPO.exists():
    subprocess.run(['git','clone','-q','https://github.com/lathrahul/modern-ai-ml-exercises.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','-q'],check=False)
sns.set_theme(style='whitegrid')
KEY=EXERCISE[:2]
CONFIG={
 '01':('exercise-01-homevalue','sample_submission.csv'),
 '02':('exercise-02-retention','sample_submission.csv'),
 '03':('exercise-03-booking-guard','sample_submission.csv'),
 '04':('exercise-04-segment-studio',None),
 '05':('exercise-05-visual-sort','sample_submission.csv'),
 '06':('exercise-06-causal-campaign','sample_effect_estimates.csv'),
}
slug,sample_name=CONFIG[KEY]
BASE=REPO/slug

In [ ]:
if USE_PUBLIC_SAMPLE:
    if sample_name is None:
        raise ValueError('Exercise 04 has no sample segmentation. Set USE_PUBLIC_SAMPLE=False and upload segments.csv.')
    output_path=BASE/sample_name
else:
    from google.colab import files
    uploaded=files.upload()
    if len(uploaded)!=1:
        raise ValueError('Upload exactly one primary output CSV.')
    output_path=Path('/content')/next(iter(uploaded))

output=pd.read_csv(output_path)
print('Loaded',output_path.name,output.shape)
display(output.head())

In [ ]:
EXPECTED={
 '01':['property_id','predicted_sale_price'],
 '02':['customer_id','churn_probability','contact_customer'],
 '03':['booking_id','cancellation_probability','require_deposit'],
 '04':['customer_id','segment_id'],
 '05':['image_id','predicted_label'],
 '06':['outcome','estimator','ate','ci_low','ci_high'],
}
assert output.columns.tolist()==EXPECTED[KEY], f'Expected columns {EXPECTED[KEY]}'
assert not output.isna().any().any(), 'Output contains missing values'
print('✓ Basic schema and missing-value checks passed')

In [ ]:
fig,ax=plt.subplots(figsize=(9,5))
if KEY=='01':
    assert output.predicted_sale_price.gt(0).all()
    sns.histplot(output.predicted_sale_price,bins=35,ax=ax)
    ax.set(title='Predicted sale-price distribution',xlabel='Predicted sale price ($)')
    display(output.predicted_sale_price.describe(percentiles=[.05,.25,.5,.75,.95]).to_frame())
elif KEY=='02':
    assert output.churn_probability.between(0,1).all()
    sns.histplot(data=output,x='churn_probability',hue='contact_customer',bins=30,ax=ax)
    ax.set(title=f'Churn probabilities | contact rate {output.contact_customer.mean():.1%}')
    display(output.groupby('contact_customer').churn_probability.agg(['count','mean','min','max']))
elif KEY=='03':
    assert output.cancellation_probability.between(0,1).all()
    context=pd.read_csv(BASE/'data/test.csv',usecols=['booking_id','hotel','average_daily_rate'])
    joined=context.merge(output,on='booking_id',validate='one_to_one')
    sns.histplot(data=joined,x='cancellation_probability',hue='require_deposit',bins=30,ax=ax)
    ax.set(title=f'Cancellation risk | deposit rate {joined.require_deposit.mean():.1%}')
    display(joined.groupby(['hotel','require_deposit']).average_daily_rate.agg(['count','median']))
elif KEY=='04':
    context=pd.read_csv(BASE/'data/customers.csv')
    joined=context.merge(output,on='customer_id',validate='one_to_one')
    shares=joined.segment_id.value_counts(normalize=True).sort_index()
    shares.plot.bar(ax=ax); ax.set(title='Segment shares',xlabel='Segment',ylabel='Share')
    display(joined.groupby('segment_id')[['recency_days','purchase_frequency','monetary_value','product_diversity','average_basket_value','return_line_rate']].median().round(2))
elif KEY=='05':
    counts=output.predicted_label.value_counts().sort_values()
    counts.plot.barh(ax=ax); ax.set(title='Predicted merchandise classes',xlabel='Images')
    archive=np.load(BASE/'data/test_images.npz'); lookup=dict(zip(archive['image_id'],archive['images']))
    sample=output.sample(min(12,len(output)),random_state=7)
    fig2,axes=plt.subplots(3,4,figsize=(10,8)); axes=np.array(axes).ravel()
    for axis,row in zip(axes,sample.itertuples()):
        axis.imshow(lookup[row.image_id],cmap='gray'); axis.set_title(row.predicted_label); axis.axis('off')
    plt.tight_layout()
elif KEY=='06':
    assert (output.ci_low<=output.ate).all() and (output.ate<=output.ci_high).all()
    y=np.arange(len(output)); xerr=np.vstack([output.ate-output.ci_low,output.ci_high-output.ate])
    ax.errorbar(output.ate,y,xerr=xerr,fmt='o',capsize=5); ax.axvline(0,color='black',lw=1)
    ax.set_yticks(y,output.outcome); ax.set(title='Estimated causal effects with confidence intervals',xlabel='ATE')
    display(output.assign(interval_crosses_zero=(output.ci_low<=0)&(output.ci_high>=0)))
plt.tight_layout(); plt.show()

## Interpretation prompts

1. What looks plausible in this output?
2. What looks surprising or potentially wrong?
3. Which operational group, tail, class, or interval should you investigate next?
4. What can this chart **not** tell you without hidden labels or a validation set?